# 06 · Power BI Export

Builds the single order-level table consumed by Power BI. The dashboard never reads raw Olist tables directly; it uses the canonical processed master table plus downstream analytical artifacts.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH=Path('../data/processed/master_orders_clean_v2.csv')
PERSONA_PATH=Path('../outputs/interpretation/order_personas.csv')
PLACEMENT_PATH=Path('../outputs/predictions/placement_predictions.csv')
DELIVERY_PATH=Path('../outputs/predictions/delivery_predictions.csv')
POWERBI_DIR=Path('../data/powerbi')
POWERBI_DIR.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(DATA_PATH,low_memory=False)


In [ ]:
keep=[
'order_id','order_purchase_timestamp','customer_state','primary_product_category',
'n_items','total_price','total_freight','total_payment_value','primary_payment_type',
'n_payment_installments','promised_delivery_days','delivery_time_days',
'delivery_vs_estimate_days','delivered_late','late_days','review_bad','split'
]
bi=df[keep].copy()
bi['order_purchase_timestamp']=pd.to_datetime(bi['order_purchase_timestamp'],errors='coerce')
bi['purchase_year']=bi['order_purchase_timestamp'].dt.year
bi['purchase_month']=bi['order_purchase_timestamp'].dt.to_period('M').astype(str)


In [ ]:
if PERSONA_PATH.exists():
    personas=pd.read_csv(PERSONA_PATH)
    bi=bi.merge(personas[['order_id','persona']],on='order_id',how='left')
else:
    bi['persona']='Not exported yet'

if PLACEMENT_PATH.exists():
    placement=pd.read_csv(PLACEMENT_PATH)
    bi=bi.merge(placement,on='order_id',how='left')

if DELIVERY_PATH.exists():
    delivery=pd.read_csv(DELIVERY_PATH)
    bi=bi.merge(delivery,on='order_id',how='left')
    prob_col=next((c for c in ['delivery_probability','predicted_probability'] if c in bi.columns),None)
    if prob_col:
        bi['risk_band']=pd.cut(bi[prob_col],bins=[-0.01,0.30,0.60,1.0],labels=['Low','Medium','High'])


The export stays useful before order-level model probabilities are added: it supports executive, delivery-risk, category, state and persona analysis. Once the original saved placement/delivery scoring artifacts are available, the same table automatically incorporates them without changing the Power BI connection.

In [ ]:
OUT=POWERBI_DIR/'powerbi_order_risk.csv'
bi.to_csv(OUT,index=False)
print('Saved:',OUT)
print('Rows:',len(bi),'Columns:',len(bi.columns))
bi.head()
